In [26]:
# Google Colab setup: fetch repository and set working directory
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_3'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Working directory: /content/BITS_programming/module_2/week_6/use_case_3


In [27]:
# AWS credentials setup via Google Colab Secrets
import os

def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region

print(f"AWS credentials loaded. Region: {_aws_region}")

AWS credentials loaded. Region: ap-south-1


In [28]:
# Install boto3/pyspark and provision dynamic S3 buckets using AWS Account ID
!pip install boto3 pyspark pandas -q
import os
import boto3
from pathlib import Path
from botocore.exceptions import ClientError

region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

_s3 = boto3.client("s3", region_name=region)

BASE_BUCKET_NAMES = ['usecase-etl-1', 'usecase-etl-2']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKET_NAMES]

for _b in REQUIRED_BUCKETS:
    try:
        if region == "us-east-1":
            _s3.create_bucket(Bucket=_b)
        else:
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"Created bucket: {_b}")
    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyExists", "BucketAlreadyOwnedByYou", "Conflict"):
            print(f"Bucket already exists (reusing existing): {_b}")
        else:
            print(f"Could not create bucket {_b} ({_code}): {_e}")

# Auto-seed raw dataset if missing from S3
RAW_BUCKET = REQUIRED_BUCKETS[0]
RAW_KEY = "churn/raw/telco_customer_churn_sample.csv"

REPO_ROOT = Path('/content/BITS_programming')
possible_raw_paths = [
    Path("../04_Datasets/raw/telco_customer_churn_sample.csv"),
    REPO_ROOT / "04_Datasets" / "raw" / "telco_customer_churn_sample.csv",
] + list(REPO_ROOT.glob("**/telco_customer_churn_sample.csv"))

local_raw = next((p for p in possible_raw_paths if p.exists()), None)

try:
    _s3.head_object(Bucket=RAW_BUCKET, Key=RAW_KEY)
    print("ℹ️ Raw data already present in S3 bucket.")
except Exception:
    if local_raw and local_raw.exists():
        print(f"📦 Seeding raw data from ({local_raw}) to s3://{RAW_BUCKET}/{RAW_KEY}...")
        _s3.upload_file(str(local_raw), RAW_BUCKET, RAW_KEY)
        print("✅ Raw dataset seeded successfully.")
    else:
        print("⚠️ Local dataset file not found for seeding.")

Bucket already exists (reusing existing): usecase-etl-1-455865672536
Bucket already exists (reusing existing): usecase-etl-2-455865672536
ℹ️ Raw data already present in S3 bucket.


# Use Case 3 - Glue ETL Job Notebook

## Converted from Glue Python job to Jupyter notebook

This notebook reads churn data directly from S3 using Boto3 + Pandas, applies ETL cleanup and feature engineering with PySpark, validates the dataset, and publishes ML-ready outputs.

In [29]:
import sys, os, boto3
from pathlib import Path

region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

BUCKET_1 = f"usecase-etl-1-{account_id}"
BUCKET_2 = f"usecase-etl-2-{account_id}"

DEFAULT_SOURCE_PATH = f"s3://{BUCKET_1}/churn/raw/telco_customer_churn_sample.csv"
DEFAULT_TARGET_PATH = f"s3a://{BUCKET_2}/ml_ready/glue_output/"

# Check runtime environment
try:
    from awsglue.context import GlueContext
    from awsglue.utils import getResolvedOptions
    from awsglue.job import Job
    HAS_GLUE = True
except ImportError:
    HAS_GLUE = False

from pyspark.sql import SparkSession
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

if HAS_GLUE and len(sys.argv) > 1 and '--JOB_NAME' in sys.argv:
    args = getResolvedOptions(sys.argv, ['JOB_NAME', 'SOURCE_PATH', 'TARGET_PATH'])
    sc = SparkContext()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    job = Job(glueContext)
    job.init(args['JOB_NAME'], args)
    print("Running in native AWS Glue environment.")
else:
    print("Running in Google Colab / Standard PySpark environment.")
    spark = SparkSession.builder \
        .appName("Glue_ETL_Notebook") \
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
        .getOrCreate()

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    hadoop_conf.set("fs.s3a.access.key", os.environ.get("AWS_ACCESS_KEY_ID", ""))
    hadoop_conf.set("fs.s3a.secret.key", os.environ.get("AWS_SECRET_ACCESS_KEY", ""))
    if os.environ.get("AWS_SESSION_TOKEN"):
        hadoop_conf.set("fs.s3a.session.token", os.environ.get("AWS_SESSION_TOKEN"))
        hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    else:
        hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    hadoop_conf.set("fs.s3a.endpoint", f"s3.{region}.amazonaws.com")

    args = {
        'JOB_NAME': 'glue_etl_colab_run',
        'SOURCE_PATH': DEFAULT_SOURCE_PATH,
        'TARGET_PATH': DEFAULT_TARGET_PATH
    }
    job = None

print(f"Source path: {args['SOURCE_PATH']}")
print(f"Target path: {args['TARGET_PATH']}")

Running in Google Colab / Standard PySpark environment.
Source path: s3://usecase-etl-1-455865672536/churn/raw/telco_customer_churn_sample.csv
Target path: s3a://usecase-etl-2-455865672536/ml_ready/glue_output/


## Step 3 - Extract: Direct S3 Read via Boto3 & Pandas

Reads directly from S3 using Boto3 into Pandas memory with `dtype=str`. This completely bypasses PySpark's JVM CSV parser to avoid `NumberFormatException` when dirty strings (e.g. `"60s"`) exist in integer or float columns.

In [30]:
import boto3, io
import pandas as pd
import numpy as np

# 1. Parse S3 source URI
s3_uri = args['SOURCE_PATH'].replace('s3a://', '').replace('s3://', '')
bucket, key = s3_uri.split('/', 1)

# 2. Extract raw data directly from S3
s3_client = boto3.client("s3", region_name=region)
obj = s3_client.get_object(Bucket=bucket, Key=key)
pdf = pd.read_csv(io.BytesIO(obj['Body'].read()), dtype=str)

# 3. Clean dirty string characters ('60s' -> 60) across numeric fields
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    if col in pdf.columns:
        pdf[col] = pdf[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)

# Safe numeric casting with automatic NaN conversion for blanks
pdf['tenure'] = pd.to_numeric(pdf['tenure'], errors='coerce')
pdf['MonthlyCharges'] = pd.to_numeric(pdf['MonthlyCharges'], errors='coerce')
pdf['TotalCharges'] = pd.to_numeric(pdf['TotalCharges'], errors='coerce')

# 4. Transform & Feature Engineering
pdf['label'] = (pdf['Churn'] == 'Yes').astype(int)
pdf['is_new_customer'] = (pdf['tenure'] <= 6).astype(int)
pdf['monthly_charge_band'] = pd.cut(
    pdf['MonthlyCharges'],
    bins=[-np.inf, 35, 70, np.inf],
    labels=['Low', 'Medium', 'High']
)
pdf['avg_monthly_spend_gap'] = pdf['TotalCharges'] - (pdf['tenure'] * pdf['MonthlyCharges'])

# 5. Load / Write output directly back to S3 target bucket
target_uri = args['TARGET_PATH'].replace('s3a://', '').replace('s3://', '')
target_bucket, target_prefix = target_uri.split('/', 1)
target_key = f"{target_prefix.rstrip('/')}/part-00000.csv"

csv_buffer = io.StringIO()
pdf.to_csv(csv_buffer, index=False)

s3_client.put_object(
    Bucket=target_bucket,
    Key=target_key,
    Body=csv_buffer.getvalue().encode('utf-8')
)

print(f"✅ Data written successfully to S3: s3://{target_bucket}/{target_key}")

✅ Data written successfully to S3: s3://usecase-etl-2-455865672536/ml_ready/glue_output/part-00000.csv


## Step 4 - Transform: Clean types and engineer features

Sanitize non-numeric characters using `regexp_replace` before casting to numeric types.

In [31]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

df = (
    df
    .withColumn('tenure', F.col('tenure').cast(IntegerType()))
    .withColumn('TotalCharges', F.col('TotalCharges').cast(DoubleType()))
    .withColumn('MonthlyCharges', F.col('MonthlyCharges').cast(DoubleType()))
    .withColumn('label',
        F.when(F.col('Churn') == 'Yes', F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn('is_new_customer',
        F.when(F.col('tenure') <= 6, F.lit(1)).otherwise(F.lit(0))
    )
    .withColumn('monthly_charge_band',
        F.when(F.col('MonthlyCharges') <= 35, F.lit('Low'))
         .when(F.col('MonthlyCharges') <= 70, F.lit('Medium'))
         .otherwise(F.lit('High'))
    )
    .withColumn(
        'avg_monthly_spend_gap',
        F.col('TotalCharges') - (F.col('tenure').cast(DoubleType()) * F.col('MonthlyCharges'))
    )
)

## Step 5 - Validate: Check data quality before writing

In [32]:
null_labels = df.filter(F.col('label').isNull()).count()
null_total_charges = df.filter(F.col('TotalCharges').isNull()).count()
row_count = df.count()

print(f"Row count          : {row_count}")
print(f"Null labels        : {null_labels}")
print(f"Null TotalCharges  : {null_total_charges}")

if null_labels > 0:
    raise ValueError(f"Validation failed: {null_labels} rows have null label. Aborting job.")

Row count          : 650
Null labels        : 0
Null TotalCharges  : 24


## Step 6 - Load: Write ML-ready output to S3 and commit

In [33]:
import io

# Convert cleaned Spark DataFrame to Pandas for direct S3 upload
output_pdf = df.toPandas()

# Extract target bucket and prefix from args
target_uri = args['TARGET_PATH'].replace('s3a://', '').replace('s3://', '')
target_bucket, target_prefix = target_uri.split('/', 1)
target_key = f"{target_prefix.rstrip('/')}/part-00000.csv"

# Buffer CSV data in memory
csv_buffer = io.StringIO()
output_pdf.to_csv(csv_buffer, index=False)

# Direct S3 upload
s3_client = boto3.client("s3", region_name=region)
s3_client.put_object(
    Bucket=target_bucket,
    Key=target_key,
    Body=csv_buffer.getvalue().encode('utf-8')
)

if job is not None:
    job.commit()
    print("Glue job committed successfully.")
else:
    print(f"✅ Data written successfully to S3 target path: s3://{target_bucket}/{target_key}")

✅ Data written successfully to S3 target path: s3://usecase-etl-2-455865672536/ml_ready/glue_output/part-00000.csv


In [34]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-14 11:47:20
